In [45]:
import requests
from urllib.parse import urlencode
import torch
import time
import cv2
import urllib.request
import numpy as np
import os

# Function to stop the robot
def stop(jetbot_ip):
    url = f'http://{jetbot_ip}:8080/stop'
    response = requests.get(url)
    
    if response.status_code == 200:
        print("Stop command executed successfully")
    else:
        print("Failed to execute stop command")

# Function to set motor speeds directly
def set_motors(jetbot_ip, left_speed, right_speed):
    params = {'left': left_speed, 'right': right_speed}
    url = f'http://{jetbot_ip}:8080/set_motors?{urlencode(params)}'
    response = requests.get(url)
    
    
def display_jetbot_video(jetbot_ip="localhost"):
    """
    Fetches and displays the video stream from the JetBot's HTTP server.
    :param jetbot_ip: The IP address of the JetBot (e.g., "192.168.1.50" or "localhost").
    """
    url = f"http://{jetbot_ip}:8080/camera"
    
    print(f"Connecting to video stream at {url}...")
    print("Press 'q' in the video window to close the stream.")
    
    while True:
        try:
            # Open the URL and read the JPEG image bytes
            with urllib.request.urlopen(url) as response:
                img_data = response.read()
                
            # Convert raw bytes to a NumPy array for OpenCV
            nparr = np.frombuffer(img_data, np.uint8)
            frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
            
            # Display the frame if it was successfully decoded
            if frame is not None:
                cv2.imshow('JetBot Video Stream', frame)
                
            # Exit loop when 'q' is pressed
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
                
        except Exception as e:
            print(f"Connection error: {e}. Retrying or check your server URL.")
            break
            
    # Clean up the display window and release resources
    cv2.destroyAllWindows()



def get_image_array(jetbot_ip="localhost"):

    url = f"http://{jetbot_ip}:8080/camera"

    with urllib.request.urlopen(url) as response:
        img_data = response.read()

    # Decode JPEG to NumPy array (BGR image)
    img_bgr = cv2.imdecode(
        np.frombuffer(img_data, np.uint8),
        cv2.IMREAD_COLOR
    )

    return img_bgr


def find_red_box(image_bgr):

    threshold_r = 120
    threshold_gb = 105


    # Channels
    R = image_bgr[:, :, 2]
    G = image_bgr[:, :, 1]
    B = image_bgr[:, :, 0]

    # Red mask
    red_mask = (R > threshold_r) & \
               (G < threshold_gb) & \
               (B < threshold_gb)

    # Find coordinates
    indices = np.argwhere(red_mask)

    if len(indices) == 0:
        return None

    # Center
    center_row, center_col = np.mean(indices, axis=0)

    # Bounding box
    min_row, min_col = np.min(indices, axis=0)
    max_row, max_col = np.max(indices, axis=0)

    # Area
    area = len(indices)

    # Distance estimate
    D = 27 * 629 / (max_row - min_row)
    
    return center_row, center_col, area, D, min_row, min_col, max_row, max_col


def blue(image_bgr):

    red_mask = (image_bgr[:, :, 0] > 110) & \
               (image_bgr[:, :, 1] < 80) & \
               (image_bgr[:, :, 2] < 30)

    # Find coordinates
    indices = np.argwhere(red_mask)

    if len(indices) == 0:
        return None

    # Center
    center_row, center_col = np.mean(indices, axis=0)

    # Bounding box
    min_row, min_col = np.min(indices, axis=0)
    max_row, max_col = np.max(indices, axis=0)

    # Area
    area = len(indices)

    D = 14.5 * 629 / (max_row - min_row)
    
    return center_row, center_col, area, D, min_row, min_col, max_row, max_col

def yellow(image_bgr):

    red_mask = (image_bgr[:, :, 0] < 80) & \
               (image_bgr[:, :, 1] > 110) & \
               (image_bgr[:, :, 1] < 130) & \
               (image_bgr[:, :, 2] > 100) & \
               (image_bgr[:, :, 2] < 140)

    # Find coordinates
    indices = np.argwhere(red_mask)

    if len(indices) == 0:
        return None

    # Center
    center_row, center_col = np.mean(indices, axis=0)

    # Bounding box
    min_row, min_col = np.min(indices, axis=0)
    max_row, max_col = np.max(indices, axis=0)

    # Area
    area = len(indices)

    D = 14.5 * 629 / (max_row - min_row)
    
    return center_row, center_col, area, D, min_row, min_col, max_row, max_col


def purple(image_bgr):

    red_mask = (image_bgr[:, :, 2] > 120) & \
               (image_bgr[:, :, 2] < 140) & \
               (image_bgr[:, :, 1] < 90) & \
                (image_bgr[:, :, 1] < 90) & \
                (image_bgr[:, :, 0] < 120) & \
               (image_bgr[:, :, 0] > 100)

    # Find coordinates
    indices = np.argwhere(red_mask)

    if len(indices) == 0:
        return None

    # Center
    center_row, center_col = np.mean(indices, axis=0)

    # Bounding box
    min_row, min_col = np.min(indices, axis=0)
    max_row, max_col = np.max(indices, axis=0)

    # Area
    area = len(indices)

    D = 14.5 * 629 / (max_row - min_row)
    
    return center_row, center_col, area, D, min_row, min_col, max_row, max_col

def green(image_bgr):

    red_mask = (image_bgr[:, :, 2] > 50) & \
               (image_bgr[:, :, 2] < 80) & \
               (image_bgr[:, :, 1] < 95) & \
                (image_bgr[:, :, 1] > 85) & \
                (image_bgr[:, :, 0] < 90) & \
               (image_bgr[:, :, 0] > 60)

    # Find coordinates
    indices = np.argwhere(red_mask)

    if len(indices) == 0:
        return None

    # Center
    center_row, center_col = np.mean(indices, axis=0)

    # Bounding box
    min_row, min_col = np.min(indices, axis=0)
    max_row, max_col = np.max(indices, axis=0)

    # Area
    area = len(indices)

    D = 14.5 * 629 / (max_row - min_row)
    
    return center_row, center_col, area, D, min_row, min_col, max_row, max_col



def draw_box(image_bgr, center_col, center_row, min_col, min_row, max_col, max_row):
    cv2.putText(
        image_bgr,
        str(np.round(D,0)),
        (int(center_col), int(center_row)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )
    
    cv2.rectangle(
        image_bgr,
        (min_col, min_row),
        (max_col, max_row),
        (0, 0, 255),   # red in BGR (correct)
        2
    )
    
    return image_bgr


In [2]:
bot = { 'bot_1': "194.47.156.201",
        'bot_2': "194.47.156.39", 
        'bot_3': '194.47.156.43',
        'bot_4': '194.47.156.213'}

jetbot_ip = bot['bot_2']

In [43]:
STOP_AREA = 49000
result = None

while True:
    left_speed, right_speed = 0.1, 0.1
    image_bgr = get_image_array(jetbot_ip)
    print(image_bgr.shape)
    key = cv2.waitKey(1)
    if key == ord('q'):
        stop(jetbot_ip)
        break
        
    result = find_red_box(image_bgr)
    result_yellow = yellow(image_bgr)

    if result is None:
        cv2.putText(
            image_bgr,
            "SEARCHING",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 255),
            2
        )


        # slow rotate
        set_motors(jetbot_ip,0.12,0.08)
        cv2.imshow("JetBot View", image_bgr)
        result = find_red_box(image_bgr)
        continue
    left_speed, right_speed = 0.15, 0.15
    set_motors(jetbot_ip, left_speed, right_speed)
    center_row, center_col, area, D, min_row, min_col, max_row, max_col = result
    
    image_bgr = draw_box(image_bgr, center_col, center_row, min_col, min_row, max_col, max_row)
    if result_yellow:
        image_bgr = draw_box(image_bgr, result_yellow)

    
    row = int(center_row)
    col = int(center_col)

    error_x = col - 500
    turn  =   error_x / (D*100)

    left_speed += turn
    right_speed -= turn
    set_motors(jetbot_ip, left_speed, right_speed)
    
    if D < 40 or area > STOP_AREA:
#         time.sleep(1.4)
        stop(jetbot_ip)
        cv2.putText(
            image_bgr,
            "STOPPED",
            (20, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 0, 255),
            2
        )
    
    cv2.imshow("JetBot View", image_bgr)
    
cv2.destroyAllWindows()

(500, 1000, 3)


QObject::moveToThread: Current thread (0x594c7c12dff0) is not the object's thread (0x594c7c26d530).
Cannot move to target thread (0x594c7c12dff0)

QObject::moveToThread: Current thread (0x594c7c12dff0) is not the object's thread (0x594c7c26d530).
Cannot move to target thread (0x594c7c12dff0)

QObject::moveToThread: Current thread (0x594c7c12dff0) is not the object's thread (0x594c7c26d530).
Cannot move to target thread (0x594c7c12dff0)

QObject::moveToThread: Current thread (0x594c7c12dff0) is not the object's thread (0x594c7c26d530).
Cannot move to target thread (0x594c7c12dff0)

QObject::moveToThread: Current thread (0x594c7c12dff0) is not the object's thread (0x594c7c26d530).
Cannot move to target thread (0x594c7c12dff0)

QObject::moveToThread: Current thread (0x594c7c12dff0) is not the object's thread (0x594c7c26d530).
Cannot move to target thread (0x594c7c12dff0)

QObject::moveToThread: Current thread (0x594c7c12dff0) is not the object's thread (0x594c7c26d530).
Cannot move to tar

(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000

In [30]:
stop(jetbot_ip)

Stop command executed successfully


In [5]:
cv2.destroyAllWindows()


### Fitness = 1 / distance

In [261]:
### Fitness = 1 / distanc

In [46]:
STOP_AREA = 52000
result = None

while True:
    left_speed, right_speed = 0.1, 0.1
    image_bgr = get_image_array(jetbot_ip)
    print(image_bgr.shape)
    key = cv2.waitKey(1)
    if key == ord('q'):
#         stop(jetbot_ip)
        break
        
    result = find_red_box(image_bgr)
    result_yellow = yellow(image_bgr)

    if result is None:
        cv2.putText(
            image_bgr,
            "SEARCHING",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 255),
            2
        )


        # slow rotate
        cv2.imshow("JetBot View", image_bgr)
        result = find_red_box(image_bgr)
        continue
    left_speed, right_speed = 0.15, 0.15
#     set_motors(jetbot_ip, left_speed, right_speed)
    center_row, center_col, area, D, min_row, min_col, max_row, max_col = result
    
    image_bgr = draw_box(image_bgr, center_col, center_row, min_col, min_row, max_col, max_row)
    if result_yellow:
        image_bgr = draw_box(image_bgr, result_yellow)

    
    row = int(center_row)
    col = int(center_col)

    error_x = col - 500
    turn  =   error_x / (D*100)

    left_speed += turn
    right_speed -= turn
#     set_motors(jetbot_ip, left_speed, right_speed)


    
    cv2.imshow("JetBot View", image_bgr)
    
cv2.destroyAllWindows()

(500, 1000, 3)


QObject::moveToThread: Current thread (0x594c7c12dff0) is not the object's thread (0x594c7c26d530).
Cannot move to target thread (0x594c7c12dff0)

QObject::moveToThread: Current thread (0x594c7c12dff0) is not the object's thread (0x594c7c26d530).
Cannot move to target thread (0x594c7c12dff0)

QObject::moveToThread: Current thread (0x594c7c12dff0) is not the object's thread (0x594c7c26d530).
Cannot move to target thread (0x594c7c12dff0)

QObject::moveToThread: Current thread (0x594c7c12dff0) is not the object's thread (0x594c7c26d530).
Cannot move to target thread (0x594c7c12dff0)

QObject::moveToThread: Current thread (0x594c7c12dff0) is not the object's thread (0x594c7c26d530).
Cannot move to target thread (0x594c7c12dff0)

QObject::moveToThread: Current thread (0x594c7c12dff0) is not the object's thread (0x594c7c26d530).
Cannot move to target thread (0x594c7c12dff0)

QObject::moveToThread: Current thread (0x594c7c12dff0) is not the object's thread (0x594c7c26d530).
Cannot move to tar

(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)


/tmp/ipykernel_1405/1695596016.py:113: RuntimeWarning: divide by zero encountered in scalar divide
  D = 27 * 629 / (max_row - min_row)


(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)
(500, 1000, 3)


/tmp/ipykernel_1405/1695596016.py:168: RuntimeWarning: divide by zero encountered in divide
  D = 14.5 * 629 / (max_row - min_row)


TypeError: draw_box() missing 5 required positional arguments: 'center_row', 'min_col', 'min_row', 'max_col', and 'max_row'